# This notebook does four things:
* necessary imports
* loading input data
* calling training function
* saving metrics

## Imports and parameters

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.datasets import fetch_openml

from utils.combined_preprocessing import preprocess_temporal_data
from utils.model_functions import model_training


In [ ]:
target_column = "count"
time_column = 'datetime'
id_column = 'season'

model_type = 'linearregression'

n_outer_splits = 5
n_inner_splits = 3
random_state = 42

## Input data

### Loading

In [ ]:
# file_path = Path("data.csv")
# inputs_df = pd.read_csv(file_path)

# Fetch the Bike Sharing Demand dataset
bike_sharing = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True)
inputs_df = bike_sharing.frame

# 1. Convert year (0 -> 2011, 1 -> 2012) and cast numeric types
inputs_df['year'] = inputs_df['year'].astype(int) + 2011
inputs_df['month'] = inputs_df['month'].astype(int)
inputs_df['hour'] = inputs_df['hour'].astype(int)

# 2. Detect when a new day starts (hour rolls back to 0 within the same month)
hour_rolled = inputs_df['hour'] < inputs_df['hour'].shift(1)
same_month = inputs_df['month'] == inputs_df['month'].shift(1)

# Increment day number for each rollover per (year, month)
inputs_df['day'] = (hour_rolled & same_month).groupby([inputs_df['year'], inputs_df['month']]).cumsum() + 1

# 3. Assemble the datetime column
inputs_df['datetime'] = pd.to_datetime(inputs_df[['year', 'month', 'day', 'hour']])

print(inputs_df.head())

### Additional preprocessing

In [ ]:
# preprocess temporal data
X, y, dates = preprocess_temporal_data(inputs_df, target_column=target_column, time_column=time_column, id_column=id_column)

print(X.head())
print(y.head())
print(dates.head())

In [ ]:
# Skewed data can be transformed using log transformation or power transform


## Model training

In [ ]:
for model_type in ['linearregression', 'ridge', 'elasticnet']:
    print(f"\nTraining model: {model_type}\n")
    model_training(inputs_df, time_column=time_column, target_column=target_column, id_column=id_column, model_type=model_type, n_outer_splits=n_outer_splits, n_inner_splits=n_inner_splits, random_state=random_state)

## Saving results

In [ ]:
...